In [7]:
import os, glob, pickle, re
import numpy as np
from extract_features import extract_features
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report

print("✅ All imports done!")

✅ All imports done!


In [8]:
EMOTIONS = {
    'a':  'angry',
    'd':  'disgust',
    'f':  'fear',
    'h':  'happy',
    'n':  'neutral',
    'sa': 'sad',
    'su': 'surprise'
}

BASE_DIR  = os.getcwd()
DATA_PATH = os.path.join(BASE_DIR, "dataset", "ALL")
print(f"📂 Loading from: {DATA_PATH}")

files = glob.glob(os.path.join(DATA_PATH, "*.wav"))
print(f"🎵 Files found: {len(files)}")

📂 Loading from: /Users/durgesh/Documents/voice to emotion/Voice-To-Emotion/dataset/ALL
🎵 Files found: 480


In [9]:
X, y = [], []

for file in files:
    basename = os.path.basename(file)
    match = re.search(r"_([a-z]{1,2})\d+\.wav", basename)
    if not match:
        continue
    code = match.group(1)
    if code not in EMOTIONS:
        continue
    features = extract_features(file_path=file)
    if features is None:
        continue
    X.append(features)
    y.append(EMOTIONS[code])

print(f"✅ Samples loaded: {len(X)}")

/Users/durgesh/Documents/voice to emotion/Voice-To-Emotion/venv/lib/python3.14/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=1024 is too large for input signal of length=961
  warnings.warn(
/Users/durgesh/Documents/voice to emotion/Voice-To-Emotion/venv/lib/python3.14/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=1024 is too large for input signal of length=765
  warnings.warn(
/Users/durgesh/Documents/voice to emotion/Voice-To-Emotion/venv/lib/python3.14/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=1024 is too large for input signal of length=896
  warnings.warn(
/Users/durgesh/Documents/voice to emotion/Voice-To-Emotion/venv/lib/python3.14/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=1024 is too large for input signal of length=781
  warnings.warn(
/Users/durgesh/Documents/voice to emotion/Voice-To-Emotion/venv/lib/python3.14/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=1024 is too large for inpu

✅ Samples loaded: 480


In [10]:
X = np.array(X)
y = np.array(y)

le     = LabelEncoder()
y_enc  = le.fit_transform(y)
scaler = StandardScaler()
X_sc   = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_sc, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)

param_grid = {
    'C':      [0.1, 1, 10, 100],
    'gamma':  ['scale', 'auto', 0.001, 0.01],
    'kernel': ['rbf', 'poly']
}

svm = SVC(class_weight='balanced', random_state=42, probability=True)
grid = GridSearchCV(svm, param_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=2)
grid.fit(X_train, y_train)

print(f" Best parameters: {grid.best_params_}")
print(f" Best CV accuracy: {grid.best_score_*100:.2f}%")

Fitting 5 folds for each of 32 candidates, totalling 160 fits
[CV] END ....................C=0.1, gamma=scale, kernel=poly; total time=   0.1s
[CV] END ....................C=0.1, gamma=scale, kernel=poly; total time=   0.1s
[CV] END .....................C=0.1, gamma=scale, kernel=rbf; total time=   0.1s
[CV] END .....................C=0.1, gamma=scale, kernel=rbf; total time=   0.1s
[CV] END .....................C=0.1, gamma=scale, kernel=rbf; total time=   0.1s
[CV] END .....................C=0.1, gamma=scale, kernel=rbf; total time=   0.1s
[CV] END ....................C=0.1, gamma=scale, kernel=poly; total time=   0.1s
[CV] END .....................C=0.1, gamma=scale, kernel=rbf; total time=   0.1s
[CV] END ....................C=0.1, gamma=scale, kernel=poly; total time=   0.1s
[CV] END ....................C=0.1, gamma=scale, kernel=poly; total time=   0.1s
[CV] END ......................C=0.1, gamma=auto, kernel=rbf; total time=   0.1s
[CV] END ......................C=0.1, gamma=aut

In [11]:
model = grid.best_estimator_
preds = model.predict(X_test)

print(f" Test Accuracy: {accuracy_score(y_test, preds)*100:.2f}%")
print(classification_report(y_test, preds, target_names=le.classes_))

pickle.dump(
    {'model': model, 'scaler': scaler, 'le': le},
    open('model.pkl', 'wb')
)
print("✅ Model saved → model.pkl")

 Test Accuracy: 80.21%
              precision    recall  f1-score   support

       angry       0.85      0.92      0.88        12
     disgust       0.62      0.67      0.64        12
        fear       0.80      0.67      0.73        12
       happy       0.80      0.67      0.73        12
     neutral       0.88      0.92      0.90        24
         sad       0.91      0.83      0.87        12
    surprise       0.71      0.83      0.77        12

    accuracy                           0.80        96
   macro avg       0.79      0.79      0.79        96
weighted avg       0.81      0.80      0.80        96

✅ Model saved → model.pkl
